# Lakehouse Data Engineering: Bronze to Silver Table Transformations

This notebook demonstrates a typical data engineering workflow in Databricks using PySpark and Delta Lake. Raw data is ingested into bronze tables, then transformed and cleansed into silver tables for analytics and downstream processing.

---

## 1. Setup and Schema Creation <a name="setup"></a>

We begin by ensuring the target schema for the silver tables exists.

-- Create the 'silver' schema if it does not already exist.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver;

## 2. Bronze to Silver Transformations <a name="transformations"></a>

### Trips Table <a name="trips"></a>

Transform the raw trips data from the bronze layer. Parse timestamps and assign meaningful column names.

In [0]:
from pyspark.sql.functions import to_timestamp, col

silver_trips = (
    spark.table("bronze.trips")
    .select(
        col("_c0").alias("ride_id"),
        col("_c1").alias("rideable_type"),
        to_timestamp("_c2").alias("started_at"),
        to_timestamp("_c3").alias("ended_at"),
        col("_c4").alias("start_station_id"),
        col("_c5").alias("end_station_id"),
        col("_c6").alias("rider_id")
    )
)

silver_trips.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.trips")

### Riders Table <a name="riders"></a>

Transform the raw riders data. Parse dates, calculate rider age at account start, and assign descriptive column names.

In [0]:
from pyspark.sql.functions import to_date, datediff, floor

silver_riders = (
    spark.table("bronze.riders")
    .select(
        col("_c0").alias("rider_id"),
        col("_c1").alias("first_name"),
        col("_c2").alias("last_name"),
        col("_c3").alias("address"),
        to_date("_c4").alias("birthday"),
        to_date("_c5").alias("account_start_date"),
        to_date("_c6").alias("account_end_date"),
        col("_c7").alias("is_member")
    )

    .withColumn(
        "age_at_account_start",
        floor(datediff("account_start_date", "birthday") / 365.25)
    )
)

silver_riders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.riders")


### Stations Table <a name="stations"></a>

Transform the raw stations data. Cast latitude and longitude to double and assign descriptive column names.

In [0]:
silver_stations = (
    spark.table("bronze.stations")
    .select(
        col("_c0").alias("station_id"),
        col("_c1").alias("station_name"),
        col("_c2").cast("double").alias("latitude"),
        col("_c3").cast("double").alias("longitude")
    )
)

silver_stations.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.stations")

### Payments Table <a name="payments"></a>

Transform the raw payments data. Parse dates, cast amount to double, and assign descriptive column names.

In [0]:
from pyspark.sql.functions import to_date

silver_payments = (
    spark.table("bronze.payments")
    .select(
        col("_c0").alias("payment_id"),
        to_date("_c1").alias("payment_date"),
        col("_c2").cast("double").alias("amount"),
        col("_c3").alias("rider_id")
    )
)

silver_payments.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.payments")

## 3. Schema Validation <a name="validation"></a>

Display the schema of each silver table to verify the transformations.

In [0]:
for t in ["trips", "riders", "stations", "payments"]:
    print(f"\nSchema silver.{t}")
    spark.table(f"silver.{t}").printSchema()


Schema silver.trips
root
 |-- trip_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- rider_id: integer (nullable = true)


Schema silver.riders
root
 |-- rider_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- birthday: date (nullable = true)
 |-- account_start_date: date (nullable = true)
 |-- account_end_date: date (nullable = true)
 |-- member_type: boolean (nullable = true)
 |-- age_at_account_start: long (nullable = true)


Schema silver.stations
root
 |-- station_id: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)


Schema silver.payments
root
 |-- payment_id: intege

---

## Conclusion

This notebook demonstrates a robust approach to data cleansing and transformation using Databricks, PySpark, and Delta Lake. The resulting silver tables are ready for analytics and further processing.